# **Import Library**

In [1]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
import joblib
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
from sklearn.ensemble import RandomForestClassifier
import scipy.stats as stats
import optuna
from sklearn.model_selection import cross_val_score
import lightgbm as lgbm

# **Load Data**

In [3]:
data = pd.read_csv("/kaggle/input/datafixxxx/dataset_training_final_FIXED.csv")
RANDOM_SEED = 42
TEST_SIZE = 0.2

# **Split**

In [4]:
X = data.drop(columns=['label', 'text', 'query', 'len_q', 'len_t'])
y = data['label']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# **Random Forest** 

In [6]:
def objective_rf(trial):    
    n_estimators = trial.suggest_int('num_estimators', 50, 100) 
    max_depth = trial.suggest_int('max_depth', 5, 20) 
    min_samples_split = trial.suggest_int('min_samples_split', 10, 30)

    modelrf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)

    score = cross_val_score(
        modelrf,
        X_train,
        y_train,
        cv=2,
        scoring='roc_auc'
    ).mean()

    return score
    

In [7]:
studyrf = optuna.create_study(direction='maximize')
studyrf.optimize(objective_rf, n_trials=100)

[I 2025-12-11 04:16:55,975] A new study created in memory with name: no-name-c62489ae-c0fd-4bd7-9f42-3e5ca9c044e7
[I 2025-12-11 04:17:14,936] Trial 0 finished with value: 0.8049995593139156 and parameters: {'num_estimators': 63, 'max_depth': 19, 'min_samples_split': 24}. Best is trial 0 with value: 0.8049995593139156.
[I 2025-12-11 04:17:29,311] Trial 1 finished with value: 0.8098507688964157 and parameters: {'num_estimators': 68, 'max_depth': 11, 'min_samples_split': 16}. Best is trial 1 with value: 0.8098507688964157.
[I 2025-12-11 04:17:50,034] Trial 2 finished with value: 0.8085269306430496 and parameters: {'num_estimators': 79, 'max_depth': 15, 'min_samples_split': 19}. Best is trial 1 with value: 0.8098507688964157.
[I 2025-12-11 04:18:00,101] Trial 3 finished with value: 0.8090398970013691 and parameters: {'num_estimators': 55, 'max_depth': 9, 'min_samples_split': 18}. Best is trial 1 with value: 0.8098507688964157.
[I 2025-12-11 04:18:14,212] Trial 4 finished with value: 0.8080

In [8]:
bestrf = studyrf.best_trial
print(f"parameter terbaik: {bestrf.params}")

parameter terbaik: {'num_estimators': 85, 'max_depth': 12, 'min_samples_split': 18}


In [9]:
parameterrf = bestrf.params
rf = RandomForestClassifier(n_estimators=parameterrf['num_estimators'], max_depth=parameterrf['max_depth'], min_samples_split=parameterrf['min_samples_split'], random_state=42)
rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=12, min_samples_split=18, n_estimators=85,
                       random_state=42)

In [10]:
y_scoresrf = rf.predict_proba(X_test)[:, 1]
map_scorerf = average_precision_score(y_test, y_scoresrf)
auc_rocrf = roc_auc_score(y_test, y_scoresrf)

print("hasil RF")
print(f"MAP : {map_scorerf}")
print(f"AUC-ROC : {auc_rocrf}")

hasil RF
MAP : 0.6708178610028814
AUC-ROC : 0.8066683747676352


In [11]:
joblib.dump(rf, "randomforest.pkl")

['randomforest.pkl']

# **XGBoost**

In [12]:
def objective_xgb(trial):    
    param = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
    }
    modelxgb = xgb.XGBClassifier(**param)

    score = cross_val_score(
        modelxgb,
        X_train,
        y_train,
        cv=2,
        scoring='roc_auc'
    ).mean()

    return score
    

In [13]:
study = optuna.create_study(direction='maximize') 
study.optimize(objective_xgb, n_trials=100, show_progress_bar=True)  

[I 2025-12-11 04:47:54,682] A new study created in memory with name: no-name-a45e8c82-2a9c-49dc-b484-3c9136069381


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-12-11 04:47:57,648] Trial 0 finished with value: 0.8101366026315302 and parameters: {'max_depth': 3, 'learning_rate': 0.07556116884592029, 'n_estimators': 679, 'subsample': 0.5184504747604732, 'colsample_bytree': 0.7488759403871943, 'min_child_weight': 6, 'gamma': 1.1592522659786486}. Best is trial 0 with value: 0.8101366026315302.
[I 2025-12-11 04:48:02,144] Trial 1 finished with value: 0.811022269926339 and parameters: {'max_depth': 7, 'learning_rate': 0.01385408355407284, 'n_estimators': 792, 'subsample': 0.5806432230483387, 'colsample_bytree': 0.838440390112489, 'min_child_weight': 8, 'gamma': 3.2138968016390512}. Best is trial 1 with value: 0.811022269926339.
[I 2025-12-11 04:48:03,433] Trial 2 finished with value: 0.8109808326862562 and parameters: {'max_depth': 5, 'learning_rate': 0.06622327088254018, 'n_estimators': 248, 'subsample': 0.6520132172430267, 'colsample_bytree': 0.9683404059765305, 'min_child_weight': 5, 'gamma': 2.4809269192560746}. Best is trial 1 with valu

In [14]:
bestxgb = study.best_trial
print(f"parameter terbaik: {bestxgb.params}")

parameter terbaik: {'max_depth': 4, 'learning_rate': 0.01818157526483247, 'n_estimators': 954, 'subsample': 0.8197283454089697, 'colsample_bytree': 0.75644961917541, 'min_child_weight': 8, 'gamma': 0.3440882009568851}


In [15]:
parameterxgb = study.best_params
xgb = xgb.XGBClassifier(**parameterxgb)
xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.75644961917541, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=0.3440882009568851,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01818157526483247,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=8, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=954, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [16]:
y_scoresxgb = xgb.predict_proba(X_test)[:, 1]
map_scorexgb = average_precision_score(y_test, y_scoresxgb)
auc_rocxgb = roc_auc_score(y_test, y_scoresxgb)

print("hasil xgb")
print(f"MAP : {map_scorexgb}")
print(f"AUC-ROC : {auc_rocxgb}")

hasil xgb
MAP : 0.6712696257066981
AUC-ROC : 0.808576199470966


In [17]:
joblib.dump(xgb, "xgboost.pkl")

['xgboost.pkl']

# **LightGBM**

In [18]:
def objective_lgbm(trial):    
    param = {
            "num_iterations": trial.suggest_int("num_iterations", 50, 150, step=10),
            "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 50, step=5),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 500, step=20),
    }
    modellgbm = lgbm.LGBMClassifier(**param)

    score = cross_val_score(
        modellgbm,
        X_train,
        y_train,
        cv=3,
        scoring="neg_log_loss"
    ).mean()

    return -score
    

In [19]:
studylgbm = optuna.create_study(direction="minimize")
studylgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

[I 2025-12-11 04:54:18,304] A new study created in memory with name: no-name-0cccfee9-5584-419b-849f-981bb94543e2


  0%|          | 0/50 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 22716, number of negative: 68148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002336 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 798
[LightGBM] [Info] Number of data points in the train set: 90864, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.250000 -> initscore=-1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Number of positive: 22716, number of negative: 68149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000635 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 800
[LightGBM] [Info] Number of data points in the train set: 90865, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.249997 -> initscore=-1.098627
[Light

In [20]:
print("Best Log Loss:", study.best_value)
print("Best Params:", study.best_params)

Best Log Loss: 0.8112359671786992
Best Params: {'max_depth': 4, 'learning_rate': 0.01818157526483247, 'n_estimators': 954, 'subsample': 0.8197283454089697, 'colsample_bytree': 0.75644961917541, 'min_child_weight': 8, 'gamma': 0.3440882009568851}


In [21]:
lightgbm = lgbm.LGBMClassifier(**study.best_params)
lightgbm.fit(X_train, y_train)

[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Info] Number of positive: 34074, number of negative: 102223
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003058 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 800
[LightGBM] [Info] Number of data points in the train set: 136297, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.249998 -> initscore=-1.098622
[LightGBM] [Info] Start training from score -1.098622
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

LGBMClassifier(colsample_bytree=0.75644961917541, gamma=0.3440882009568851,
               learning_rate=0.01818157526483247, max_depth=4,
               min_child_weight=8, n_estimators=954,
               subsample=0.8197283454089697)

In [22]:
joblib.dump(lightgbm, "lightgbm.pkl")

['lightgbm.pkl']